In [2]:
# B6_4 — Step 2.2 (diagnostic, CPU): why does the FSC-147 gate fail? Reads rows only; no counts are changed.
import json, numpy as np, pandas as pd
from pathlib import Path
from google.colab import drive; drive.mount('/content/drive')
R = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results")
def load(d):
    out = {}
    for p in sorted(d.glob("*.json")):
        r = json.load(open(p)); out.setdefault(r["key"], r)
    return out
om = load(R / "B6_4_Baseline_OmniCount" / "fsc147" / "rows_omnicount")
tf = load(R / "B6_3_Baselines_SAM_TFOC" / "fsc147" / "rows_tfoc")
rows = []
for k, r in om.items():
    c = r["classes"][0]; d = r["diag"]["per_class"]["OmniCount"][c]
    rows.append({"key": k, "class": c, "y": r["meta"]["gt"][c], "omni": r["counts"]["OmniCount"][c],
                 "tfoc": tf[k]["counts"]["TFOC"][c] if k in tf else np.nan, "points": d["n_points"],
                 "fallback": d["fallback"], "san_empty": d["san_empty"]})
df = pd.DataFrame(rows)                                             # GT column is "y" ("gt" clashes with DataFrame.gt)
df["e_omni"] = df["omni"] - df["y"]; df["e_tfoc"] = df["tfoc"] - df["y"]
print(f"images {len(df)} | TFOC rows matched {df['tfoc'].notna().sum()} | MAE OmniCount {df['e_omni'].abs().mean():.2f} | TFOC {df['e_tfoc'].abs().mean():.2f}")
bins = [0, 1, 5, 10, 20, 50, 100, 200, np.inf]; labels = ["1", "2-5", "6-10", "11-20", "21-50", "51-100", "101-200", ">200"]
df["bin"] = pd.cut(df["y"], bins, labels=labels)
tot = df["e_omni"].abs().sum()
def agg(g):
    return pd.Series({"n": len(g), "MAE_omni": g["e_omni"].abs().mean(), "bias_omni": g["e_omni"].mean(),
                      "MAE_tfoc": g["e_tfoc"].abs().mean(), "bias_tfoc": g["e_tfoc"].mean(),
                      "share_|e|_omni_%": 100 * g["e_omni"].abs().sum() / tot, "median_points/gt": (g["points"] / g["y"]).median()})
print("\n(1) By GT count bin (bias = mean signed error; negative = undercount)")
print(pd.DataFrame({b: agg(g) for b, g in df.groupby("bin", observed=True)}).T.round(2).to_string())
print("\n(2) By fallback / empty SAN mask")
print(pd.DataFrame({k: agg(g) for k, g in df.groupby(["fallback", "san_empty"])}).T.round(2).to_string())
print("\n(3) Images with omni < gt:", round(100 * (df["e_omni"] < 0).mean(), 1), "% | tfoc < gt:", round(100 * (df["e_tfoc"] < 0).mean(), 1), "%",
      "| OmniCount worse than TFOC on", int((df["e_omni"].abs() > df["e_tfoc"].abs()).sum()), "of", len(df), "images")
order = df["e_omni"].abs().sort_values(ascending=False).index
top = df.loc[order[:15]]
print(f"\n(4) Top 15 |error| images (they hold {100 * top['e_omni'].abs().sum() / tot:.1f}% of total |error|)")
print(top[["key", "class", "y", "omni", "tfoc", "points", "fallback", "san_empty"]].rename(columns={"y": "gt"}).to_string(index=False))
rest = df.loc[order[10:]]
print(f"\n(5) MAE without the 10 largest-|error| images: OmniCount {rest['e_omni'].abs().mean():.2f} | TFOC on the same images {rest['e_tfoc'].abs().mean():.2f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
images 1190 | TFOC rows matched 1190 | MAE OmniCount 29.34 | TFOC 24.70

(1) By GT count bin (bias = mean signed error; negative = undercount)
             n  MAE_omni  bias_omni  MAE_tfoc  bias_tfoc  share_|e|_omni_%  median_points/gt
6-10      60.0      8.07       7.13      5.83       3.03              1.39              7.35
11-20    268.0      8.77       5.86      6.74       2.80              6.73              5.16
21-50    413.0     12.54       4.07     10.21       0.07             14.83              2.98
51-100   254.0     23.48      -5.08     19.24      -3.55             17.08              1.60
101-200  139.0     53.73     -38.73     42.37     -28.67             21.39              0.88
>200      56.0    240.50    -232.71    218.59    -203.27             38.57              0.38

(2) By fallback / empty SAN mask
                       n  MAE_omni  bias_om